# 揽宝多数据源演示

本笔记本演示如何使用揽宝平台的多数据源适配器：
- Tushare适配器
- 通达信(TDX)适配器
- AkShare适配器
- MiniQMT适配器

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
from lanbao_data import TushareAdapter, TDXAdapter, AKShareAdapter, MiniQMTAdapter
import pandas as pd

# 设置环境变量
os.environ['TUSHARE_TOKEN'] = 'your_tushare_token_here'  # 如果需要使用Tushare

## 1. Tushare 适配器演示

In [ ]:
# 初始化Tushare适配器
try:
    tushare_adapter = TushareAdapter()
    print(f"Tushare适配器优先级: {tushare_adapter.priority}")
    print(f"Tushare可用性: {tushare_adapter.is_available()}")
except Exception as e:
    print(f"Tushare初始化失败: {e}")

In [ ]:
# 获取日线数据
symbol = '000001.SZ'
start_date = '20240101'
end_date = '20241231'

try:
    df = tushare_adapter.get_daily_data(symbol, start_date, end_date)
    print(f"获取到 {len(df)} 条数据")
    print(df.head())
except Exception as e:
    print(f"获取数据失败: {e}")

## 2. 通达信(TDX) 适配器演示

In [ ]:
# 初始化通达信适配器
try:
    tdx_adapter = TDXAdapter()
    print(f"通达信适配器优先级: {tdx_adapter.priority}")
    print(f"通达信可用性: {tdx_adapter.is_available()}")
except Exception as e:
    print(f"通达信初始化失败: {e}")

In [ ]:
# 获取日线数据
try:
    df = tdx_adapter.get_daily_data(symbol, start_date, end_date)
    print(f"获取到 {len(df)} 条数据")
    print(df.head())
except Exception as e:
    print(f"获取数据失败: {e}")

In [ ]:
# 获取实时行情
try:
    quote = tdx_adapter.get_realtime_quote(symbol)
    print(f"实时行情: {quote}")
except Exception as e:
    print(f"获取实时行情失败: {e}")

## 3. AkShare 适配器演示

In [ ]:
# 初始化AkShare适配器
try:
    akshare_adapter = AKShareAdapter()
    print(f"AkShare适配器优先级: {akshare_adapter.priority}")
    print(f"AkShare可用性: {akshare_adapter.is_available()}")
except Exception as e:
    print(f"AkShare初始化失败: {e}")

In [ ]:
# 获取日线数据
try:
    df = akshare_adapter.get_daily_data(symbol, start_date, end_date)
    print(f"获取到 {len(df)} 条数据")
    print(df.head())
except Exception as e:
    print(f"获取数据失败: {e}")

In [ ]:
# 获取股票列表
try:
    stocks = akshare_adapter.get_stock_list('A')
    print(f"A股股票数量: {len(stocks)}")
    print(stocks.head(10))
except Exception as e:
    print(f"获取股票列表失败: {e}")

In [ ]:
# 获取股票新闻 (AkShare特有功能)
try:
    news = akshare_adapter.get_stock_news(symbol)
    print(f"获取到 {len(news)} 条新闻")
    print(news.head())
except Exception as e:
    print(f"获取新闻失败: {e}")

## 4. MiniQMT 适配器演示

In [ ]:
# 初始化MiniQMT适配器
# 注意: 需要先安装QMT客户端并配置QMT_PATH环境变量
try:
    miniqmt_adapter = MiniQMTAdapter()
    print(f"MiniQMT适配器优先级: {miniqmt_adapter.priority}")
    print(f"MiniQMT可用性: {miniqmt_adapter.is_available()}")
except Exception as e:
    print(f"MiniQMT初始化失败: {e}")

## 5. 多数据源优先级对比

In [ ]:
# 创建所有适配器并比较
adapters = []

for name, AdapterClass in [
    ('Tushare', TushareAdapter),
    ('通达信', TDXAdapter),
    ('AkShare', AKShareAdapter),
    ('MiniQMT', MiniQMTAdapter)
]:
    try:
        adapter = AdapterClass()
        adapters.append({
            '名称': name,
            '适配器': adapter,
            '优先级': adapter.priority,
            '可用': adapter.is_available()
        })
    except Exception as e:
        adapters.append({
            '名称': name,
            '适配器': None,
            '优先级': '-',
            '可用': False
        })

comparison_df = pd.DataFrame(adapters)
print(comparison_df[['名称', '优先级', '可用']])

## 6. 数据源配置示例

可以通过环境变量配置要使用的数据源：

In [ ]:
# 配置示例
import os

# 启用多个数据源（按优先级排序）
os.environ['LANBAO_DATA_SOURCES'] = 'miniqmt,tushare,akshare,tdx'

print("环境变量配置:")
print(f"LANBAO_DATA_SOURCES = {os.environ.get('LANBAO_DATA_SOURCES', 'tushare,akshare')}")
print(f"\n数据源优先级（数字越小优先级越高）:")
print("MiniQMT: 1 (实盘交易数据源，优先级最高)")
print("Tushare: 1 (高质量数据源)")
print("通达信: 2 (快速实时行情)")
print("AkShare: 3 (免费数据源，作为fallback)")

## 7. 数据对比分析

对比不同数据源获取的数据差异：

In [ ]:
# 从多个数据源获取同一股票的数据进行对比
symbol = '000001.SZ'
start_date = '20240101'
end_date = '20240131'

results = {}

for name, adapter in [('Tushare', tushare_adapter), ('AkShare', akshare_adapter)]:
    try:
        df = adapter.get_daily_data(symbol, start_date, end_date)
        if not df.empty:
            results[name] = df
    except Exception as e:
        print(f"{name} 获取失败: {e}")

# 对比数据
for name, df in results.items():
    print(f"\n{name}:")
    print(f"  数据条数: {len(df)}")
    print(f"  日期范围: {df['date'].min()} ~ {df['date'].max()}")
    print(f"  最新收盘价: {df['close'].iloc[-1]:.2f}")